### Import

In [22]:
# ===== standard libs =====
import os, sys, platform, random
from pathlib import Path

# ===== core libs =====
import numpy as np
import pandas as pd

# ===== vision / augmentation =====
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ===== ML =====
import torch
import timm
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix

# ===== utilities =====
from tqdm import tqdm
import yaml
import matplotlib.pyplot as plt
import wandb

print("✅ All imports OK")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))

print("timm:", timm.__version__)
print("albumentations:", A.__version__)
print("opencv:", cv2.__version__)


✅ All imports OK
Python: 3.10.13
Platform: Linux-5.4.0-166-generic-x86_64-with-glibc2.31
Torch: 2.7.1+cu118
CUDA available: True
CUDA device: NVIDIA GeForce RTX 3090
timm: 1.0.24
albumentations: 2.0.8
opencv: 4.13.0


### Set Route

In [23]:
# notebook 위치 기준
notebook_dir = Path.cwd()

# case 1: notebook_dir == cvproject/model
project_root = notebook_dir.parent

# case 2: notebook_dir == cvproject/model/notebooks
# project_root = notebook_dir.parent.parent

data_dir = project_root / "data"
train_dir = data_dir / "train"
test_dir  = data_dir / "test"

model_dir = project_root / "Model"
artifacts_dir = model_dir / "artifacts"
ckpt_dir = artifacts_dir / "ckpts"
oof_dir  = artifacts_dir / "oof"

submission_dir = project_root / "Submission"
sub_csv_dir = submission_dir / "submissions"

for p in [artifacts_dir, ckpt_dir, oof_dir, sub_csv_dir]:
    p.mkdir(parents=True, exist_ok=True)

print("project_root:", project_root)
print("train_dir exists:", train_dir.exists())
print("test_dir exists:", test_dir.exists())
print("ckpt_dir:", ckpt_dir)
print("sub_csv_dir:", sub_csv_dir)

project_root: /root/CVProject
train_dir exists: True
test_dir exists: True
ckpt_dir: /root/CVProject/Model/artifacts/ckpts
sub_csv_dir: /root/CVProject/Submission/submissions


### Data Load

In [24]:
train_csv = data_dir / "train.csv"
meta_csv  = data_dir / "meta.csv"
sample_sub_csv = data_dir / "sample_submission.csv"

train_df = pd.read_csv(train_csv)
print("train_df shape:", train_df.shape)
display(train_df.head(4))

assert "ID" in train_df.columns and "target" in train_df.columns
assert str(train_df.loc[0, "ID"]).endswith(".jpg")

# 파일 존재 체크 (랜덤 10개)
sample_ids = train_df["ID"].sample(10, random_state=42).tolist()
missing = [fid for fid in sample_ids if not (train_dir / fid).exists()]
print("missing files:", missing)


train_df shape: (1570, 2)


,ID,target
0,002f99746285dfdd.jpg,16
1,008ccd231e1fea5d.jpg,10
2,008f5911bfda7695.jpg,10
3,009235e4c9c07af5.jpg,4


missing files: []


### Config

In [25]:
def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["pythonhashseed"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

cfg = {
    "seed": 42,
    "num_folds": 5,
    "img_size": 384,
    "batch_size": 16,
    "num_workers": 4,
    "epochs": 15,

    "model_name": "convnext_base",
    "pretrained": True,
    "dropout": 0.0,

    "optimizer": "adamw",
    "lr": 1e-4,
    "weight_decay": 0.05,

    "use_amp": True,
    "tta": {"enabled": False},

    "paths": {
        "train_csv": str(train_csv),
        "meta_csv": str(meta_csv),
        "sample_sub": str(sample_sub_csv),
        "train_dir": str(train_dir),
        "test_dir": str(test_dir),
        "ckpt_dir": str(ckpt_dir),
        "oof_dir": str(oof_dir),
        "sub_dir": str(sub_csv_dir),
    },

    "wandb": {
        "enabled": True,
        "project": "cv-doctype-classification",
        "entity": None,
        "run_name": "convnext_base_kfold5_img384_base"
    }
}

seed_everything(cfg["seed"])
print("✅ config ready")


✅ config ready


### W&B Check

In [26]:
print("wandb api key set:", "WANDB_API_KEY" in os.environ)

# 최초 1회
# !wandb login

print("✅ wandb ready")


wandb api key set: False
✅ wandb ready
